# Progressive Student Model Training

Trains a split pipeline for distilling our Claude-based extraction pipeline into local models:
- **NER**: BioLinkBERT-large (340M) with BIOES tagging + sliding window
- **RE**: BioLinkBERT-large (340M) pair classifier with typed entity markers

Trained at progressive scales: 1 → 10 → 100 → 400 BioRED gold examples.

**Runtime**: Select GPU (Runtime → Change runtime type → T4 GPU)

## Setup

In [ ]:
# Mount Google Drive (optional — for persisting results across sessions)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repo and install dependencies (torch is pre-installed on Colab)
!git clone https://github.com/Currie32/biolink-hub.git /content/biolink-hub
%cd /content/biolink-hub
!pip install -e '.[train]' -q

In [ ]:
# Verify GPU
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(name)s %(levelname)s %(message)s",
)

## Quick smoke test (N=1, N=10)

Run the split pipeline at the two smallest scales to validate everything works.
Expected time: ~30 min on T4 (BioLinkBERT-large is 340M params).

In [ ]:
from bioextract.model.train_progressive import run_progressive

results_small = run_progressive(scales=[1, 10])

### Smoke test checks

- **N=1**: The pipeline should memorize the single example
- **N=10**: NER should tag at least some entities on unseen text; RE should predict a few relationships

In [ ]:
import json
print(json.dumps(results_small, indent=2))

## Full experiment (N=100, N=400)

Run the larger scales. Expected time: ~5-6 hr on T4 (BioLinkBERT-large is 340M params).

You can run these individually if you're worried about session timeouts.

In [ ]:
results_large = run_progressive(scales=[100, 400])

## Review results

In [ ]:
# Load the full results file (includes all scales run so far)
import json
from pathlib import Path

results_path = Path("models/progressive_results.json")
with open(results_path) as f:
    all_results = json.load(f)

print(json.dumps(all_results, indent=2))

In [ ]:
from bioextract.model.train_progressive import _print_summary
_print_summary(all_results)

## Copy results to Drive (optional)

In [ ]:
import shutil
from pathlib import Path

drive_dest = Path("/content/drive/MyDrive/biolink-hub/models")
drive_dest.mkdir(parents=True, exist_ok=True)

# Copy results JSON
shutil.copy("models/progressive_results.json", drive_dest / "progressive_results.json")

# Copy best model (adjust path based on results)
# shutil.copytree("models/ner_n400", drive_dest / "ner_n400", dirs_exist_ok=True)
# shutil.copytree("models/re_n400", drive_dest / "re_n400", dirs_exist_ok=True)

print(f"Results copied to {drive_dest}")